# Silver Layer Processing - Drug Data
## Healthcare Lakehouse Project

This notebook transforms the raw Bronze provider-drug data into a clean **Silver layer** with drug dimension and fact tables.

### Transformations Applied:
1. **Data Quality Checks**: Null analysis, duplicate detection
2. **Type Casting**: Convert columns to appropriate types
3. **Standardization**: Uppercase and trim text values, clean drug names
4. **Deduplication**: Aggregate duplicates by key columns
5. **Derived Columns**: Calculate cost per claim
6. **Data Modeling**: Create drug dimension and fact tables

### Output Tables:
- **silver.drug**: Drug dimension table
- **silver.by_provider_drug**: Provider-drug fact table

### Step 1: Import Required Libraries

In [ ]:
# Databricks notebook source
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### Step 2: Load Utility Variables and Configure Widgets
Runs the utilities notebook and sets up runtime configuration.

In [ ]:
# MAGIC %run "/Workspace/Users/pawanvirat32@gmail.com/healthcare_lakehouse_project/01_setup/utilities"

In [ ]:
print(silver_schema)

dbutils.widgets.text('catalog', 'healthcare_lakehouse', 'Catalog')
dbutils.widgets.text('data_source', 'by_provider_drug', 'Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

print(f"catalog: {catalog}")
print(f"data_source: {data_source}")

### Step 3: Configure Paths and Load Bronze Data
Sets up S3 paths and reads data from the Bronze layer.

In [ ]:
S3_BASE        = f"s3://healthcare-claims-lakehouse/raw/{data_source}"
CHECKPOINT_BASE = f"s3://healthcare-claims-lakehouse/checkpoints/{data_source}"
TARGET_SCHEMA  = f"{catalog}.silver"

In [ ]:
df_bronze = spark.sql(f'select * from {catalog}.{bronze_schema}.{data_source}')

### Step 4: Preview Bronze Data
Displays sample records to understand the data structure.

In [ ]:
display(df_bronze.limit(10))

In [ ]:
df_bronze.select(
    "brnd_name",
    "gnrc_name"
).show(10)

### Step 5: Select Relevant Columns
Selects key columns for the fact table.

In [ ]:
df = df_bronze.select(
    "prscrbr_npi",
    "brnd_name",
    "gnrc_name",
    "tot_clms",
    "tot_drug_cst",
    "tot_benes"
)

### Step 6: Create Drug Dimension Table
Extracts distinct drug names and assigns unique drug_id.

In [ ]:
df_drug = df_bronze.select("gnrc_name") \
    .filter(F.col("gnrc_name").isNotNull()) \
    .dropDuplicates() \
    .withColumn("drug_id", F.monotonically_increasing_id())

In [ ]:
df_drug.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.{silver_schema}.drug")

### Step 7: Preview Fact Data

In [ ]:
display(df.limit(10))

In [ ]:
df.printSchema()

### Step 8: Data Quality - Check Nulls
Analyzes null values across all columns.

In [ ]:
df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

In [ ]:
df.select(
    F.count("*").alias("total_rows"),
    F.count(F.col("tot_benes")).alias("non_null_benes")
).show()

### Step 9: Add Missing Flag Column
Creates a flag to track records with missing beneficiaries data.

In [ ]:
df = df.withColumn(
    "benes_missing_flag",
    F.when(F.col("tot_benes").isNull(), 1).otherwise(0)
)

In [ ]:
df.printSchema()

### Step 10: Type Casting
Casts numeric columns to appropriate types.

In [ ]:
df = df.withColumn("tot_clms", F.col("tot_clms").cast("int")) \
       .withColumn("tot_drug_cst", F.col("tot_drug_cst").cast("double")) \
       .withColumn("tot_benes", F.col("tot_benes").cast("int"))

In [ ]:
df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ["tot_clms", "tot_drug_cst", "tot_benes"]
]).show()

### Step 11: Statistical Summary
Provides descriptive statistics for numeric columns.

In [ ]:
df.describe(["tot_clms", "tot_drug_cst"]).show()

### Step 12: Duplicate Analysis
Checks for duplicate records based on key columns.

In [ ]:
df.count()

In [ ]:
df.dropDuplicates(["prscrbr_npi", "gnrc_name"]).count()

In [ ]:
df.groupBy("prscrbr_npi", "gnrc_name") \
  .count() \
  .orderBy(F.desc("count")) \
  .show(10)

### Step 13: Aggregate Duplicates
Aggregates duplicate records by summing numeric values.

In [ ]:
df = df.groupBy("prscrbr_npi", "gnrc_name") \
    .agg(
        F.first("brnd_name").alias("brnd_name"),
        F.sum("tot_clms").alias("tot_clms"),
        F.sum("tot_drug_cst").alias("tot_drug_cst"),
        F.sum("tot_benes").alias("tot_benes"),
        F.max("benes_missing_flag").alias("benes_missing_flag")
    )

In [ ]:
df.count()

In [ ]:
df.show(10)

In [ ]:
df.groupBy("prscrbr_npi", "gnrc_name").count().filter("count > 1").show()

### Step 14: Standardize Drug Names
Cleans and standardizes drug name formatting.

In [ ]:
df.select("gnrc_name").distinct().show(20, False)

In [ ]:
df = df.withColumn(
    "gnrc_name",
    F.upper(F.trim(F.col("gnrc_name")))
).withColumn(
    "brnd_name",
    F.upper(F.trim(F.col("brnd_name")))
)

In [ ]:
# "Potassium,mag" → "Potassium, mag"
df = df.withColumn(
    "gnrc_name",
    F.regexp_replace(F.col("gnrc_name"), ",\\s*", ", ")
)

In [ ]:
df.select("gnrc_name").distinct().show(20, False)

### Step 15: Create Derived Column - Cost per Claim
Calculates the average cost per claim as a key metric.

In [ ]:
df = df.withColumn(
    "cost_per_claim",
    F.when(
        F.col("tot_clms") > 0,
        F.col("tot_drug_cst") / F.col("tot_clms")
    ).otherwise(0)
)

In [ ]:
df.select("cost_per_claim").describe().show()

### Step 16: Write Final Silver Table

In [ ]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(f"{catalog}.{silver_schema}.by_provider_drug")
)

In [ ]:
print("drug:", df_drug.count())